# Wearable Device Analysis

### Data initialization

In [ ]:
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git
!pip install --quiet "git+https://github.com/mugalan/classical-mechanics-from-a-geometric-point-of-view.git#egg=rigid-body-sim"


In [1]:
from google.colab import drive
import open_dataset_store as ds
import numpy as np
import pandas as pd
import plotly.express as px
import sims
print('Open Dataset Store Version :',ds.__version__)

drive.mount('/content/drive')
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"

store = ds.quick_start(dataset_path, backend='local')


Open Dataset Store Version : 2.0.1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)


In [2]:
\sdasdsdf

📊 Dataset Store Summary
Base Directory : /content/drive/MyDrive/Datasets/Wearable-Biomechanics
Backend        : local
------------------------------
Entities (Total: 3)
  - subjects: 3
------------------------------
Entries (Total: 7)
  - experiments: 6
  - device: 1


,entry_id,entity_id,timestamp,description,raw_csv_path,processed_files
0,entry_0001,None,1781954406,"Datasets taken to check device resolution, sen...",raw_data/device/entry_0001_none_1781856275.csv,{}


## Data Cleanup

In [3]:
df = store.get_entry_raw_data("device", "entry_0001")


In [ ]:
summary = store.get_df_summary(df, detailed=True)

In [ ]:
assoc_matrix = store.plot_all_associations_heatmap(df=df)

In [4]:
df = df.iloc[0:50000]

In [5]:
def get_sensor_data(df, device_prefix):
    accel_cols = [f'{device_prefix}_AX', f'{device_prefix}_AY', f'{device_prefix}_AZ']
    gyro_cols = [f'{device_prefix}_GX', f'{device_prefix}_GY', f'{device_prefix}_GZ']

    # Normalize accelerometer data
    accel = df[accel_cols].to_numpy()
    accel_norm = np.linalg.norm(accel, axis=1, keepdims=True) + 1e-8
    y_data = accel / accel_norm

    result_df = df[gyro_cols].copy()
    result_df.columns = ['GX', 'GY', 'GZ']

    # Add the normalized accel data as new columns
    result_df[f'AX'] = y_data[:, 0]
    result_df[f'AY'] = y_data[:, 1]
    result_df[f'AZ'] = y_data[:, 2]

    result_df[f'Timestamp'] = df[f'Timestamp']

    return result_df

# Usage changes to expect a single return variable
device4_df = get_sensor_data(df, 'device4_IMU1')
device7_df = get_sensor_data(df, 'device7_IMU1')
device4_df.head()

,GX,GY,GZ,AX,AY,AZ,Timestamp
0,0.122,-0.457,0.457,-0.016630,0.999859,0.002446,1
1,0.091,-0.396,0.457,-0.017117,0.999851,0.002348,2
2,0.061,-0.427,0.518,-0.016738,0.999858,0.001860,3
3,0.122,-0.457,0.457,-0.016138,0.999868,0.001663,4
4,0.061,-0.518,0.427,-0.016540,0.999857,0.003523,5


In [6]:
fig = px.line(
    df,
    x='Timestamp',
    y=['device4_IMU1_AX','device4_IMU1_AY','device4_IMU1_AZ'],
    title='Device 1 IMU Accelerometer Data Over Time'
)

fig.show(renderer="colab")

In [7]:
# Filter Configuration
E = np.array([[0.0, 0.0, -1.0]])
dt = 0.01;
# Tuning parameters
Sigma_gyro = (0.02**2) * np.eye(3)   # Gyro noise
Sigma_bias = (1e-5**2) * np.eye(3)   # Gyro bias random walk
Sigma_q = np.block([
    [Sigma_gyro, np.zeros((3, 3))],
    [np.zeros((3, 3)), Sigma_bias],
])
Sigma_m = (0.15**2) * np.eye(3)

In [8]:
# 3. Initialize and run EKF for Device 4 (Upper Arm)
kf_4 = sims.LinearKF(use_joseph=True, symmetrize=True)
iekf_4 = sims.SO3IMUSensorFusionBiasEKF(kf=kf_4, inertial_directions=E, dt=dt)

res_4 = iekf_4.run_filter(
    R0_plus=np.eye(3),
    b0_plus=np.zeros(3),
    P0=np.diag([1e-2, 1e-2, 1e-2, 1e-4, 1e-4, 1e-4]),
    Omegas=device4_df[['GX', 'GY', 'GZ']].to_numpy(),
    Ys=device4_df[['AX', 'AY', 'AZ']].to_numpy(),
    Sigma_q=Sigma_q,
    Sigma_m=Sigma_m
)
R4_est = res_4["R_plus"] # Shape: (T, 3, 3)
